# From raw hospitalization exports to case pickles

This standalone notebook builds the unsuffixed weekly and monthly hospitalization case pickles directly from the raw weekly JSON exports. It creates no intermediate data files and does not join weather, pollution, or other drivers. The 30 outputs are written to the locations consumed by this repository: `data/weekly/raw/` and `data/monthly/raw/`.

For each hospital it retains the first diagnosis attached to each encounter, matching the existing pipeline. General counts include every encounter; ILI and HW counts include only encounters whose retained diagnosis matches the corresponding ICD-9 ontology. Monthly counts are obtained by distributing every weekly count uniformly over its seven calendar days and summing by calendar month.

In [2]:
import gc
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display


def find_repo_root(start=Path.cwd()):
    """Find the repository root whether Jupyter starts here or in a parent folder."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/ps_dare").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current directory.")


def find_source_root(repo_root):
    """Locate the legacy raw exports, with an environment override for other setups."""
    configured = os.getenv("PS_DARE_SOURCE_ROOT")
    candidates = ([Path(configured).expanduser()] if configured else []) + [
        repo_root,
        repo_root.parent / "ps-dare",
    ]
    for candidate in candidates:
        if (candidate / "data/hospitalization/weekly-aggregation/json-dump").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the raw hospitalization exports. Set PS_DARE_SOURCE_ROOT "
        "to the legacy data repository."
    )


ROOT = find_repo_root()
load_dotenv(ROOT / ".env")
SOURCE_ROOT = find_source_root(ROOT)
DATA_ROOT = Path(os.getenv("DATA_BASE_DIR", ROOT)).expanduser() / "data"
RAW_DIR = SOURCE_ROOT / "data/hospitalization/weekly-aggregation/json-dump"
HW_ONTOLOGY_PATH = SOURCE_ROOT / "data/disease_ontology/HW/ontology_heatwave.csv"
ILI_ONTOLOGY_PATH = SOURCE_ROOT / "data/disease_ontology/ILI/sorted_uniqued_full_manualv3.csv"

WRITE_OUTPUTS = False

RAW_PATTERNS = {
    "giustiniani": {
        "encounters": "ps-adulti-giustiniani-srigato-*.json",
        "diagnoses": "ps-adulti-giustiniani-dettaglio-diagnosi-*.json",
    },
    "osa": {
        "encounters": "ps-adulti-osa-srigato-*.json",
        "diagnoses": "ps-adulti-osa-dettaglio-diagnosi-*.json",
    },
    "pediatrico": {
        "encounters": "ps-pediatrico-srigato-*.json",
        "diagnoses": "ps-pediatrico-dettaglio-diagnosi-*.json",
    },
}

print(f"Repository root: {ROOT}")
print(f"Raw-data source: {SOURCE_ROOT}")
print(f"Pickle destination: {DATA_ROOT}")
print(f"Write outputs: {WRITE_OUTPUTS}")

Repository root: /Users/tommasobertola/Git/ps-dare-paper-2
Raw-data source: /Users/tommasobertola/Git/ps-dare
Pickle destination: /Users/tommasobertola/Git/ps-dare-paper-2/data
Write outputs: False


## Load and normalize the disease ontologies

Ontology codes are converted to five-character strings so they match `DIA_CODICE` in the raw diagnosis exports. Duplicate mappings are handled like the existing notebooks: ILI mappings are deduplicated by data code, while HW rows are retained.

In [3]:
def normalize_numeric_icd9(series):
    numeric = pd.to_numeric(series, errors="coerce").astype("Int64")
    return numeric.astype("string").str.zfill(5)


def load_hw_ontology(path):
    ontology = pd.read_csv(path)
    ontology.columns = ontology.columns.str.replace("'", "", regex=False)
    ontology["ICD9_DATA_CODE"] = normalize_numeric_icd9(ontology["ICD9_DATA_CODE"])
    ontology = ontology.dropna(subset=["ICD9_DATA_CODE"]).copy()
    ontology["diagnosis_category"] = (
        ontology["CCS CATEGORY DESCRIPTION"].astype("string").str.replace("'", "", regex=False)
    )
    return ontology[["ICD9_DATA_CODE", "diagnosis_category"]]


def load_ili_ontology(path):
    ontology = pd.read_csv(path)
    ontology["ICD9_DATA_CODE"] = normalize_numeric_icd9(ontology["ICD9_DATA_CODE"])
    ontology = ontology.dropna(subset=["ICD9_DATA_CODE", "diagnosis_category"]).copy()
    ontology = ontology.drop_duplicates(subset="ICD9_DATA_CODE", keep="first")
    return ontology[["ICD9_DATA_CODE", "diagnosis_category"]]


ONTOLOGIES = {
    "ili": load_ili_ontology(ILI_ONTOLOGY_PATH),
    "hw": load_hw_ontology(HW_ONTOLOGY_PATH),
}

pd.DataFrame(
    {
        name: {
            "rows": len(ontology),
            "unique_codes": ontology["ICD9_DATA_CODE"].nunique(),
            "categories": ontology["diagnosis_category"].nunique(),
        }
        for name, ontology in ONTOLOGIES.items()
    }
).T

,rows,unique_codes,categories
ili,47,47,18
hw,67,66,6


## Transformation functions

In [4]:
def resolve_one(directory, pattern):
    matches = sorted(directory.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one file matching {pattern!r} in {directory}; found {len(matches)}: {matches}"
        )
    return matches[0]


def load_encounters_with_first_diagnosis(encounters_path, diagnoses_path):
    encounters = pd.read_json(encounters_path, lines=True, dtype=False)
    diagnoses = pd.read_json(diagnoses_path, lines=True, dtype=False)

    encounters["Settimana Accesso"] = pd.to_datetime(
        encounters["Settimana Accesso"], format="%m/%d/%y %H:%M:%S"
    )

    # Preserve source order and retain one diagnosis per encounter, as in weekly-dataset.ipynb.
    first_diagnosis = diagnoses.drop_duplicates(
        subset="ID_DEA_ANONIMIZZATA", keep="first"
    )
    merged = encounters.merge(
        first_diagnosis,
        on="ID_DEA_ANONIMIZZATA",
        how="left",
        validate="one_to_one",
    )
    merged = merged.rename(columns={"Settimana Accesso": "week"})
    return merged.sort_values("week").reset_index(drop=True)


def make_weekly_cases(encounters):
    outputs = {
        "general": encounters.groupby("week").size().reset_index(name="n_accesses")
    }

    for kind, ontology in ONTOLOGIES.items():
        matched = encounters.merge(
            ontology, left_on="DIA_CODICE", right_on="ICD9_DATA_CODE", how="inner"
        )
        outputs[kind] = matched.groupby("week").size().reset_index(name="n_accesses")
        outputs[f"{kind}_cat"] = (
            matched.groupby(["week", "diagnosis_category"])
            .size()
            .reset_index(name="n_accesses")
        )

    return outputs


def split_weekly_counts_across_months(weekly, group_columns=()):
    """Allocate 1/7 of a weekly count to each day, then sum by calendar month."""
    repeated = weekly.loc[weekly.index.repeat(7)].copy()
    repeated["day_offset"] = np.tile(np.arange(7), len(weekly))
    repeated["month"] = (
        repeated["week"] + pd.to_timedelta(repeated["day_offset"], unit="D")
    ).dt.to_period("M").dt.to_timestamp()
    repeated["n_accesses"] = repeated["n_accesses"] / 7.0

    keys = ["month", *group_columns]
    return repeated.groupby(keys, as_index=False)["n_accesses"].sum()


def complete_category_months(monthly):
    months = pd.date_range(monthly["month"].min(), monthly["month"].max(), freq="MS")
    categories = monthly["diagnosis_category"].drop_duplicates().sort_values()
    full_index = pd.MultiIndex.from_product(
        [months, categories], names=["month", "diagnosis_category"]
    )
    return (
        monthly.set_index(["month", "diagnosis_category"])
        .reindex(full_index, fill_value=0.0)
        .reset_index()
    )


def make_monthly_cases(weekly_outputs):
    monthly_outputs = {}
    for name, weekly in weekly_outputs.items():
        is_category = name.endswith("_cat")
        group_columns = ["diagnosis_category"] if is_category else []
        monthly = split_weekly_counts_across_months(weekly, group_columns)
        if is_category:
            monthly = complete_category_months(monthly)

        monthly["month"] = monthly["month"].dt.strftime("%Y-%m")
        monthly["month_index"] = pd.to_datetime(monthly["month"]).dt.month_name()
        monthly_outputs[name] = monthly

        weekly_total = weekly["n_accesses"].sum()
        monthly_total = monthly["n_accesses"].sum()
        if not np.isclose(weekly_total, monthly_total):
            raise AssertionError(
                f"Count conservation failed for {name}: weekly={weekly_total}, monthly={monthly_total}"
            )

    return monthly_outputs


def output_path(frequency, hospital, name):
    kind = name.removesuffix("_cat")
    suffix = "_cat" if name.endswith("_cat") else ""
    return DATA_ROOT / frequency / "raw" / f"{kind}_{hospital}{suffix}.pkl"


def write_case_pickles(frequency, hospital, outputs):
    written = []
    for name, dataframe in outputs.items():
        path = output_path(frequency, hospital, name)
        path.parent.mkdir(parents=True, exist_ok=True)
        dataframe.to_pickle(path)
        written.append(path)
    return written

## Build all weekly and monthly pickles

Hospitals are processed one at a time to keep peak memory use down. Only the 30 final unsuffixed case pickles are written: 5 case series for each of 3 hospitals at 2 frequencies.

In [5]:
summary_rows = []
written_paths = []

for hospital, patterns in RAW_PATTERNS.items():
    encounters_path = resolve_one(RAW_DIR, patterns["encounters"])
    diagnoses_path = resolve_one(RAW_DIR, patterns["diagnoses"])
    print(f"Processing {hospital}: {encounters_path.name}")

    encounters = load_encounters_with_first_diagnosis(encounters_path, diagnoses_path)
    weekly_outputs = make_weekly_cases(encounters)
    monthly_outputs = make_monthly_cases(weekly_outputs)

    for frequency, outputs in (("weekly", weekly_outputs), ("monthly", monthly_outputs)):
        if WRITE_OUTPUTS:
            written_paths.extend(write_case_pickles(frequency, hospital, outputs))

        time_column = "week" if frequency == "weekly" else "month"
        for name, dataframe in outputs.items():
            summary_rows.append(
                {
                    "hospital": hospital,
                    "frequency": frequency,
                    "series": name,
                    "rows": len(dataframe),
                    "start": dataframe[time_column].min(),
                    "end": dataframe[time_column].max(),
                    "total_accesses": dataframe["n_accesses"].sum(),
                }
            )

    del encounters, weekly_outputs, monthly_outputs
    gc.collect()

expected_paths = {
    output_path(frequency, hospital, name)
    for frequency in ("weekly", "monthly")
    for hospital in RAW_PATTERNS
    for name in ("general", "ili", "ili_cat", "hw", "hw_cat")
}
if WRITE_OUTPUTS and set(written_paths) != expected_paths:
    raise AssertionError("Written pickle inventory does not match the repository inputs.")

summary = pd.DataFrame(summary_rows)
display(summary)
print(f"Wrote {len(written_paths)} final pickle files." if WRITE_OUTPUTS else "Dry run: no files written.")

Processing giustiniani: ps-adulti-giustiniani-srigato-02_01_2012-29_03_2026.json
Processing osa: ps-adulti-osa-srigato-02_01_2012-29_03_2026.json
Processing pediatrico: ps-pediatrico-srigato-02_01_2012-29_03_2026.json


,hospital,frequency,series,rows,start,end,total_accesses
0,giustiniani,weekly,general,744,2012-01-02 00:00:00,2026-03-30 00:00:00,1257292.0
1,giustiniani,weekly,ili,744,2012-01-02 00:00:00,2026-03-30 00:00:00,75772.0
2,giustiniani,weekly,ili_cat,7573,2012-01-02 00:00:00,2026-03-30 00:00:00,75772.0
3,giustiniani,weekly,hw,743,2012-01-02 00:00:00,2026-03-23 00:00:00,38100.0
4,giustiniani,weekly,hw_cat,3860,2012-01-02 00:00:00,2026-03-23 00:00:00,38100.0
5,giustiniani,monthly,general,172,2012-01,2026-04,1257292.0
6,giustiniani,monthly,ili,172,2012-01,2026-04,75772.0
7,giustiniani,monthly,ili_cat,3096,2012-01,2026-04,75772.0
8,giustiniani,monthly,hw,171,2012-01,2026-03,38100.0
9,giustiniani,monthly,hw_cat,1026,2012-01,2026-03,38100.0


Dry run: no files written.


## Output inventory

In [ ]:
if WRITE_OUTPUTS:
    for path in sorted(written_paths):
        try:
            print(path.relative_to(ROOT))
        except ValueError:
            print(path)